# 02 — Exploratory Data Analysis & KPI Baseline

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.features import add_delivery_features, add_route_features
from src.kpis import compute_kpi_summary, plot_daily_late_rate, plot_kpi_summary

sns.set_theme(style='whitegrid')
%matplotlib inline

PROCESSED = Path('..') / 'data' / 'processed'
orders    = pd.read_csv(PROCESSED / 'orders_clean.csv', parse_dates=['order_date','promised_date','actual_delivery_date'])
inventory = pd.read_csv(PROCESSED / 'inventory_clean.csv', parse_dates=['date'])
routes    = pd.read_csv(PROCESSED / 'routes_clean.csv', parse_dates=['date'])


## 2.1 Feature engineering

In [ ]:
orders = add_delivery_features(orders)
routes = add_route_features(routes)
print(orders[['on_time','late_delivery_flag','lead_time_hours','delivery_delay_hours']].describe())


## 2.2 KPI baseline

In [ ]:
kpi_df = compute_kpi_summary(orders, inventory, routes)
print(kpi_df.to_string(index=False))
plot_kpi_summary(kpi_df)


## 2.3 Daily late-delivery rate

In [ ]:
plot_daily_late_rate(orders)

## 2.4 Delivery lead-time distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
sns.histplot(orders['lead_time_hours'].dropna(), bins=40, ax=ax, color='steelblue')
ax.set_title('Distribution of Delivery Lead Time (hours)')
ax.set_xlabel('Lead Time (hours)')
plt.tight_layout()
plt.savefig('../outputs/figures/lead_time_dist.png', dpi=150)
plt.show()


## 2.5 Late-delivery rate by day of week

In [ ]:
dow_late = orders.groupby('day_of_week')['late_delivery_flag'].mean().reset_index()
dow_late['day_name'] = dow_late['day_of_week'].map({0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'})
fig, ax = plt.subplots(figsize=(8,4))
sns.barplot(data=dow_late, x='day_name', y='late_delivery_flag', ax=ax, palette='coolwarm')
ax.set_title('Late-Delivery Rate by Day of Week')
ax.set_ylabel('Late Rate')
plt.tight_layout()
plt.savefig('../outputs/figures/late_rate_by_dow.png', dpi=150)
plt.show()


## 2.6 Correlation heatmap (numeric features)

In [ ]:
numeric_cols = orders.select_dtypes(include=np.number).columns.tolist()
corr = orders[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(12,10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap — Orders')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150)
plt.show()


## 2.7 Stockout rate by warehouse

In [ ]:
if 'stockout_flag' in inventory.columns and 'warehouse_id' in inventory.columns:
    wh_stockout = inventory.groupby('warehouse_id')['stockout_flag'].mean().reset_index()
    fig, ax = plt.subplots(figsize=(8,4))
    sns.barplot(data=wh_stockout, x='warehouse_id', y='stockout_flag', ax=ax, palette='Reds')
    ax.set_title('Stockout Rate by Warehouse')
    ax.set_ylabel('Stockout Rate')
    plt.tight_layout()
    plt.savefig('../outputs/figures/stockout_by_warehouse.png', dpi=150)
    plt.show()
else:
    print("inventory_clean.csv missing 'stockout_flag' or 'warehouse_id' — skip.")
